In [1]:
import pandas as pd
import re

In [2]:
def transpose_columns(df,fixed_cols,sep_col = "."):
    '''
    Retourne le dataframe avec les colonnes non fixes pivotées

            Parameters:
                   df: dataframe source
                   fixed_cols: colonnes fixes à conserver
                   sep_col: le séparateur entre le nom de colonne et le numéro (ex: "." si la colonne est "Colonne.1")

            Returns:
                    result: le dataframe final avec les colonnes pivotées
    '''
    # Colonnes répétées
    repeated_cols = [c for c in df.columns if c not in fixed_cols]

    sep_escaped = re.escape(sep_col)
    # Récupération des noms de base
    base_names = sorted({
        re.sub(rf"{sep_escaped}\d+$", "", col)
        for col in repeated_cols
    })
    
    # Nombre de groupes
    nb_groups = max(
        int(re.search(rf"{sep_escaped}(\d+)$", c).group(1))
        if re.search(rf"{sep_escaped}(\d+)$", c)
        else 0
        for c in repeated_cols
    ) + 1
    
    dfs = []
    
    for i in range(nb_groups):
    
        cols = {}
        for base in base_names:
    
            col_name = base if i == 0 else f"{base}{sep_col}{i}"
    
            if col_name in df.columns:
                cols[col_name] = base
    
        tmp = df[fixed_cols].copy()
    
        for old_name, new_name in cols.items():
            tmp[new_name] = df[old_name]
    
        dfs.append(tmp)
    
    result = pd.concat(dfs, ignore_index=True)
    result = result.dropna(
        subset=base_names,
        how="all"
    )
    return result

In [3]:
def pct_cols(df):
    '''
    transforme les colonnes contenant "%" en float

            Parameters:
                    df: dataframe source

            Returns:
                    df: le dataframe transformé
    '''
    pct_cols = [c for c in df.columns if "%" in c]

    for col in pct_cols:
        df[col] = (
            pd.to_numeric(
                df[col]
                    .astype(str)
                    .str.replace("%", "", regex=False)
                    .str.replace(",", ".", regex=False),
                errors="coerce"
            )
        )
    return df

In [4]:
def col_to_bool(df,col,val_col):
    '''
    transforme les colonnes en booléen avec correspondance de valeurs

            Parameters:
                    df: dataframe source
                    col: la colonne à transformer
                    val_col: la valeur pour True (False correspondant toujours à vide)

            Returns:
                    df: le dataframe transformé
    '''
    df[col] = df[col].map({val_col:True ,None:False})
    return df

In [5]:
def harmonize_sex(df,col):
    '''
    harmonise les colonnes sexe en F/M

            Parameters:
                    df: dataframe source
                    col: la colonne à transformer

            Returns:
                    df: le dataframe transformé
    '''
    df[col] = df[col].map({'FEMININ':'F' ,'MASCULIN':'M'})
    return df

# Elections législatives 2002

## Candidatures

In [6]:
fichier_candidature_2002_t1 = "2002/sources/legislatives-2002-candidatures-france-entiere-tour-1-2024-06-21-10h50.csv"
candidature_2002_t1 = pd.read_csv(fichier_candidature_2002_t1, sep = ";")

In [7]:
fichier_candidature_2002_t2 = "2002/sources/legislatives-2002-candidatures-france-entiere-tour-2-2024-06-21-10h52.csv"
candidature_2002_t2 = pd.read_csv(fichier_candidature_2002_t2, sep = ";")

## Résultats

In [8]:
fichier_resultats_2002 = "2002/sources/96f64b379471595907b6abb0267e7b9e36a1d9ccd06521e377c8330c3130de.xls"
# Colonnes fixes
fixed_cols = [
    "Code du département", "Libellé du département", "Code de la circonscription", "Libellé de la circonscription", "Inscrits", "Abstentions", "% Abs/Ins", "Votants", "% Vot/Ins", "Blancs et nuls", "% BlNuls/Ins", "% BlNuls/Vot", "Exprimés", "% Exp/Ins", "% Exp/Vot"
]

In [9]:
resultats_2002_t1 = pd.read_excel(fichier_resultats_2002, sheet_name="Circo leg T1")
resultats_2002_t1_tr = transpose_columns(resultats_2002_t1,fixed_cols)
resultats_2002_t1_tr.to_csv("2002/work/resultats_2022_t1.csv", index=False)

In [10]:
resultats_2002_t2 = pd.read_excel(fichier_resultats_2002, sheet_name="Circo leg T2")
resultats_2002_t2_tr = transpose_columns(resultats_2002_t2,fixed_cols)
resultats_2002_t2_tr.to_csv("2002/work/resultats_2022_t2.csv", index=False)

# Elections législatives 2024

## Candidatures

In [11]:
fichier_candidature_2024_t1 = "2024/sources/legislatives-2024-candidatures-france-entiere-tour-1-2024-06-28.csv"
candidature_2024_t1 = pd.read_csv(fichier_candidature_2024_t1, sep = ";")

In [12]:
fichier_candidature_2024_t2 = "2024/sources/legislatives-2024-candidatures-france-entiere-tour-2-2024-07-05.csv"
candidature_2024_t2 = pd.read_csv(fichier_candidature_2024_t2, sep = ";")

## Résultats

In [13]:
# Colonnes fixes
fixed_cols = [
    "Code département","Libellé département","Code circonscription législative","Libellé circonscription législative","Inscrits","Votants","% Votants","Abstentions","% Abstentions","Exprimés","% Exprimés/inscrits","% Exprimés/votants","Blancs","% Blancs/inscrits","% Blancs/votants","Nuls","% Nuls/inscrits","% Nuls/votants"
]

In [14]:
fichier_2024_t1 = "2024/sources/resultats-definitifs-par-circonscriptions-legislatives.csv"
resultats_2024_t1 = pd.read_csv(fichier_2024_t1, sep = ";", decimal=",")
resultats_2024_t1 = pct_cols(resultats_2024_t1)
resultats_2024_t1_tr = transpose_columns(resultats_2024_t1,fixed_cols," ")
resultats_2024_t1_tr.to_csv("2024/work/resultats__2024_t1.csv", index=False)

In [15]:
fichier_2024_t2 = "2024/sources/resultats-definitifs-par-circonscription.csv"
resultats_2024_t2 = pd.read_csv(fichier_2024_t2, sep = ";", decimal=",",quotechar='"')
resultats_2024_t2 = pct_cols(resultats_2024_t2)
resultats_2024_t2_tr = transpose_columns(resultats_2024_t2,fixed_cols," ")
resultats_2024_t2_tr.to_csv("2024/work/resultats__2024_t2.csv", index=False)

# Préparation des données

## Renommage des colonnes

In [16]:
col_to_rename = {
    'Code département':'code_departement',
    'Code du département':'code_departement',
                     
    'Libellé du département':'lbl_departement',
    'Département':'lbl_departement',
    'Libellé département':'lbl_departement',

    'Code de la circonscription':'code_circonscription',
    'Code circonscription':'code_circonscription',
    'Code circonscription législative':'code_circonscription',

    'Libellé circonscription':'lbl_circonscription',
    'Nom_de_la_Circonscription':'lbl_circonscription',
    'Libellé de la circonscription':'lbl_circonscription',
    'Libellé circonscription législative':'lbl_circonscription',

    'Numéro de panneau':'num_panneau',
    'N° dépôt':'num_depot',

    'Nom du candidat':'nom_candidat',
    'Nom':'nom_candidat',
    'Nom candidat':'nom_candidat',

    'Prénom du candidat':'prenom_candidat',
    'Prénom':'prenom_candidat',
    'Prénom candidat':'prenom_candidat',

    'Date de naissance du candidat':'date_naissance_candidat',

    'Sexe du candidat':'sexe_candidat',
    'Sexe candidat':'sexe_candidat',
    'Sexe':'sexe_candidat',
    
    'Sortant':'bool_sortant',

    'Profession':'profession',

    'Nom remplaçant':'nom_remplacant',
    'Prénom remplaçant':'prenom_remplacant',
    'Date de naissance remplaçant':'date_naissance_remplacant',
    'Sexe remplaçant':'sexe_remplacant',
    'Sortant remplaçant':'bool_sortant_remplacant',

    'Code nuance':'code_nuance',
    'Nuance':'code_nuance',
    'Nuance candidat':'code_nuance',
    'Nuance du candidat':'lbl_nuance',

    '% Abs/Ins':'pct_abstentions_inscrits',
    '% Abstentions':'pct_abstentions',
    '% Blancs/inscrits':'pct_blancs_inscrits',
    '% Blancs/votants':'pct_blancs_votants',
    '% BlNuls/Ins':'pct_blancsnuls_inscrits',
    '% BlNuls/Vot':'pct_blancsnuls_votants',
    '% Nuls/inscrits':'pct_nuls_inscrits',
    '% Nuls/votants':'pct_nuls_votants',
    '% Exp/Ins':'pct_exprimes_inscrits',
    '% Exp/Vot':'pct_exprimes_votants',
    '% Exprimés/inscrits':'pct_exprimes_inscrits',
    '% Exprimés/votants':'pct_exprimes_votants',
    '% Voix/Exp':'pct_voix_exprimes',
    '% Voix/exprimés':'pct_voix_exprimes',
    '% Voix/Ins':'pct_voix_inscrits',
    '% Voix/inscrits':'pct_voix_inscrits',
    '% Vot/Ins':'pct_votants_inscrits',
    '% Votants':'pct_votants',

    'Abstentions':'nb_abstentions',
    'Blancs':'nb_blancs',
    'Blancs et nuls':'nb_blancsnuls',
    'Exprimés':'nb_exprimes',
    'Inscrits':'nb_inscrits',
    'Nuls':'nb_nuls',
    'Voix':'nb_voix',
    'Votants':'nb_votants',
    
    'Elu':'bool_elu'
                }

In [17]:
candidature_2002_t1.rename(columns=col_to_rename, inplace = True)
candidature_2002_t2.rename(columns=col_to_rename, inplace = True)

resultats_2002_t1_tr.rename(columns=col_to_rename, inplace = True)
resultats_2002_t2_tr.rename(columns=col_to_rename, inplace = True)

candidature_2024_t1.rename(columns=col_to_rename, inplace = True)
candidature_2024_t2.rename(columns=col_to_rename, inplace = True)

resultats_2024_t1_tr.rename(columns=col_to_rename, inplace = True)
resultats_2024_t2_tr.rename(columns=col_to_rename, inplace = True)

## Transformation des booléens

In [18]:
candidature_2002_t1 = col_to_bool(candidature_2002_t1,'bool_sortant','OUI')
candidature_2002_t1 = col_to_bool(candidature_2002_t1,'bool_sortant_remplacant','OUI')

candidature_2002_t2 = col_to_bool(candidature_2002_t2,'bool_sortant','OUI')
candidature_2002_t2 = col_to_bool(candidature_2002_t2,'bool_sortant_remplacant','OUI')

candidature_2024_t1 = col_to_bool(candidature_2024_t1,'bool_sortant','OUI')
candidature_2024_t1 = col_to_bool(candidature_2024_t1,'bool_sortant_remplacant','OUI')

candidature_2024_t2 = col_to_bool(candidature_2024_t1,'bool_sortant','OUI')
candidature_2024_t2 = col_to_bool(candidature_2024_t1,'bool_sortant_remplacant','OUI')

resultats_2024_t1_tr = col_to_bool(resultats_2024_t1_tr,'bool_elu','élu')
resultats_2024_t2_tr = col_to_bool(resultats_2024_t2_tr,'bool_elu','élu')

## Harmonisation des valeurs

### Sexe

In [22]:
resultats_2024_t1_tr = harmonize_sex(resultats_2024_t1_tr,'sexe_candidat')
resultats_2024_t2_tr = harmonize_sex(resultats_2024_t2_tr,'sexe_candidat')

### Noms et Prénoms

In [ ]:
# TO DO : passer les noms en full maj, et les prénoms en nom propre (majuscule à la première lettre)

## Calcul de l'âge

In [25]:
# TO DO : calculer l'âge en fonction du jour du tour 1 de chaque année, puis vérifier que chaque participant a plus de 18 ans et moins de 150

# Tables de travail

In [21]:
# TO DO : Créer une table candidat avec un id généré pour chaque trio nom/prénom/date de naissance